# LeetCode 14 — Longest Common Prefix | Nhập môn Python, giảng từng bước

**Độ khó:** Easy · **Kỹ thuật trọng tâm:** String · duyệt ký tự · prefix  
**Bài gốc:** https://leetcode.com/problems/longest-common-prefix/

Notebook này dành cho người **mới bắt đầu**. Đọc các ô từ trên xuống: trước hết hiểu **vì sao**, sau đó chạy code nhỏ để tự thấy **cách làm**.

**Khung hàm (signature) cần nộp LeetCode:**

```python
class Solution:
    def longestCommonPrefix(self, strs: list[str]) -> str:
        ...
```

**Sau bài học, bạn có thể:**
- Hiểu tiền tố chung và vị trí ký tự trong chuỗi.
- Tự tìm được cách thử đơn giản rồi tối ưu thành so sánh theo từng cột.
- Tính được thời gian O(n·L), bộ nhớ bổ sung O(1), và xử lý cả chuỗi không có tiền tố chung.

**Cách sử dụng:** chạy từng code cell theo thứ tự bằng `Shift + Enter`. Những ô có `TODO` là bài tự luyện: chỉ định nghĩa hàm hoặc chứa lời nhắc kiểm thử, **không được gọi tự động** khi chưa làm xong.

> Lộ trình: hiểu đề → Python căn bản → cách trực giác → phát hiện nút thắt → cải tiến → trace → nộp LeetCode → test → lỗi → vận dụng → bài tập → đáp án → checklist.

## 1. Đề bài — hỏi chính xác điều gì?

Cho danh sách `strs` gồm các chuỗi. Hãy tìm **chuỗi dài nhất nằm ở đầu của tất cả các chuỗi**; nếu không có, trả `""`.

| strs | kết quả | vì sao |
|---|---|---|
| `["flower", "flow", "flight"]` | `"fl"` | cả ba đều mở đầu bằng `f`, `l` |
| `["dog", "racecar", "car"]` | `""` | ký tự đầu tiên khác nhau |
| `["interspecies", "interstellar", "interstate"]` | `"inters"` | cả ba bắt đầu bằng `inters` |

**Không phải** tìm một đoạn xuất hiện ở bất cứ đâu (substring). `"low"` nằm trong `"flower"`, nhưng không bắt đầu ở vị trí 0 nên không phải prefix.

In [ ]:
examples = [["flower", "flow", "flight"], ["dog", "racecar", "car"]]
for words in examples:
    print("Danh sách:", words)

## 2. Python cơ sở: list, chuỗi và chỉ số

`strs` là **list** (danh sách) các **string** (chuỗi ký tự). Python đánh số vị trí từ `0` (**index**). `s[i]` đọc ký tự tại vị trí `i`; `len(s)` trả số ký tự.

Ví dụ `s = "flow"`: `s[0]` là `"f"`, `s[1]` là `"l"` và `len(s) = 4`. Nếu đọc `s[4]`, chương trình gặp `IndexError`: **vượt biên**.

In [ ]:
s = "flow"
print("Ký tự đầu:", s[0]); print("Ký tự thứ hai:", s[1]); print("Độ dài:", len(s))
words = ["flower", "flow", "flight"]
print("Chuỗi đầu tiên:", words[0]); print("Số chuỗi:", len(words))

## 3. Cú pháp lấy đoạn đầu `s[:k]`

**Slicing** là lấy một đoạn chuỗi: `s[start:end]`. `end` là **vị trí dừng, không lấy ký tự ở đó**. Nếu bỏ `start`, Python mặc định từ đầu.

Ví dụ: `"flower"[:2]` là `"fl"`; `[:0]` là chuỗi rỗng `""`. Cú pháp này sẽ hữu ích để lấy ra tiền tố đã kiểm chứng.

In [ ]:
s = "flower"
for k in [0, 1, 2, 4, len(s)]:
    print(f"s[:{k}] = {s[:k]!r}")

## 4. `startswith` nghĩa là gì?

`s.startswith(prefix)` trả về `True` nếu chuỗi `s` bắt đầu bằng `prefix`, ngược lại `False`. Đây là cách Python tự kiểm tra một tiền tố; hãy quan sát trước khi dùng trong vòng lặp.

In [ ]:
print("flower".startswith("flo"))  # True
print("flight".startswith("flo"))  # False
print("flower".startswith("low"))  # False
print("flower".startswith(""))     # True

## 5. Cách trực giác — thu ngắn ứng viên

Lấy chuỗi đầu làm **prefix ứng viên**. Với từng chuỗi kế tiếp, nếu nó không bắt đầu bằng ứng viên, ta bớt một ký tự **ở cuối ứng viên** rồi thử lại. Cuối cùng ứng viên là phần chung.

Ví dụ `["flower", "flow", "flight"]`: ban đầu `"flower"` → `"flow"` (khớp chuỗi 2) → `"flo"` → `"fl"` (khớp chuỗi 3).

In [ ]:
def lcp_shrink(strs: list[str]) -> str:
    if not strs:   # mở rộng: danh sách rỗng, dù đề chính không cần
        return ""
    prefix = strs[0]
    for word in strs[1:]:
        while not word.startswith(prefix):
            prefix = prefix[:-1]
            if prefix == "":
                return ""
    return prefix

print(lcp_shrink(["flower", "flow", "flight"]))
print(lcp_shrink(["dog", "racecar", "car"]))

## 6. Khám phá nhược điểm của cách thu ngắn

`prefix = prefix[:-1]` tạo **chuỗi mới**; `startswith` có thể phải so lại nhiều ký tự. Cách này dễ nghĩ, nhưng tạo và kiểm tra các tiền tố lặp lại. Liệu ta có thể **chỉ kiểm tra mỗi vị trí đúng một lượt**, rồi dừng ngay khi có khác biệt?

In [ ]:
prefix = "flower"
for word in ["flow", "flight"]:
    print("So với:", word)
    while not word.startswith(prefix):
        prefix = prefix[:-1]
        print("  Thu ngắn:", repr(prefix))
print("Đáp án:", repr(prefix))

## 7. Insight — kiểm tra theo cột

Cứ xét ký tự thứ `i` của **chuỗi đầu tiên** rồi so với ký tự cùng vị trí trong **mọi** chuỗi còn lại. Nếu thấy một chuỗi ngắn hơn (không có `i`) **hoặc** ký tự khác, các vị trí sau không thể thuộc tiền tố chung.

| i | `flower[i]` | `flow[i]` | `flight[i]` | quyết định |
|---:|---|---|---|---|
| 0 | f | f | f | giữ `f` |
| 1 | l | l | l | giữ `l` |
| 2 | o | o | i | **dừng**, trả `"fl"` |

**Bất biến (invariant):** trước mỗi lần kiểm tra cột `i`, toàn bộ cột từ `0` đến `i-1` đều giống nhau.

In [ ]:
strs = ["flower", "flow", "flight"]
for i in range(3):
    chars = [word[i] for word in strs]
    print("Cột", i, "->", chars, "| cùng nhau?", len(set(chars)) == 1)

## 8. Cú pháp `for`, `range`, `enumerate`

`range(k)` sinh các chỉ số `0, 1, ..., k-1`; `for x in ...` làm lần lượt từng phần tử. `enumerate(s)` sinh cặp `(index, character)`, tránh tự tăng biến đếm.

Vì cột `i` được lấy từ chuỗi đầu tiên, `enumerate(strs[0])` đảm bảo `i` luôn **hợp lệ với chuỗi đầu**. Với những chuỗi khác, vẫn phải kiểm tra `i >= len(word)` trước khi đọc `word[i]`.

In [ ]:
for i in range(3):
    print("range ->", i)
for i, ch in enumerate("cat"):
    print("enumerate -> i:", i, "ký tự:", ch)

## 9. Code chính — so sánh theo cột

Thực hiện nguyên ý tưởng ở bảng trace: tại mỗi `i`, nếu có `word` thiếu cột `i` hoặc khác `ch`, trả ngay phần đầu `strs[0][:i]`. Nếu đi hết chuỗi đầu mà không gặp khác biệt, **toàn bộ chuỗi đầu** là tiền tố chung.

In [ ]:
def longest_common_prefix_vertical(strs: list[str]) -> str:
    if not strs:
        return ""
    first = strs[0]
    for i, ch in enumerate(first):
        for word in strs[1:]:
            if i >= len(word) or word[i] != ch:
                return first[:i]
    return first

print(longest_common_prefix_vertical(["flower", "flow", "flight"]))
print(longest_common_prefix_vertical(["dog", "racecar", "car"]))

## 10. Vì sao điều kiện `or` an toàn?

Trong Python, `A or B` có tính chất **short-circuit**: nếu `A` đúng, Python **không tính** `B` nữa.

Do đó `i >= len(word) or word[i] != ch` **không bị IndexError** khi `word` quá ngắn. Đảo thứ tự hai vế là lỗi thường gặp.

In [ ]:
word = "a"
i = 2
print("Quá biên:", i >= len(word))
print("Biểu thức an toàn:", i >= len(word) or word[i] != "x")

## 11. Trace từng vòng lặp — nhìn thấy lý do dừng

Trace phải ghi cột, chữ đang so, nhánh kiểm tra độ dài và nhánh so sánh. Đây là bản **debug để học**, không phải code submit.

In [ ]:
def trace_lcp(strs):
    if not strs:
        print("Danh sách rỗng -> ''"); return ""
    first = strs[0]
    for i, ch in enumerate(first):
        print(f"\nCỘT {i}: kỳ vọng {ch!r}")
        for word in strs[1:]:
            if i >= len(word):
                print(f"  {word!r}: không có chỉ số {i} -> dừng")
                return first[:i]
            print(f"  {word!r}: word[{i}]={word[i]!r}, khớp={word[i] == ch}")
            if word[i] != ch:
                print("  KHÁC -> dừng, trả", repr(first[:i]))
                return first[:i]
    return first

assert trace_lcp(["flower", "flow", "flight"]) == "fl"

## 12. Code nộp LeetCode

LeetCode tự tạo `Solution` và gọi phương thức `longestCommonPrefix` với đúng tên. **Chỉ dán lớp này** khi nộp; không cần những hàm `print` và trace ở trên.

In [ ]:
class Solution:
    def longestCommonPrefix(self, strs: list[str]) -> str:
        if not strs:
            return ""
        first = strs[0]
        for i, ch in enumerate(first):
            for word in strs[1:]:
                if i >= len(word) or word[i] != ch:
                    return first[:i]
        return first

print("Submit demo:", Solution().longestCommonPrefix(["interspace", "internet", "internal"]))

## 13. Kiểm thử — mẫu, biên và phản ví dụ

Dùng `assert actual == expected` để Python **báo lỗi** ngay nếu kết quả sai. Test không chỉ xem đầu ra; còn phải kiểm tra trường hợp khác biệt ngay đầu, chuỗi đầu ngắn nhất, tất cả giống nhau.

In [ ]:
tests_14 = [
    (["flower", "flow", "flight"], "fl"),
    (["dog", "racecar", "car"], ""),
    (["interspecies", "interstellar", "interstate"], "inters"),
    (["a"], "a"), (["", "a"], ""), (["abc", "abc"], "abc"),
    (["abcd", "ab"], "ab"), (["a", "ab", "abc"], "a"),
]
for words, expected in tests_14:
    result = Solution().longestCommonPrefix(words)
    assert result == expected, (words, result, expected)
    assert lcp_shrink(words) == result
print("PASS", len(tests_14), "ca kiểm thử + đối chiếu 2 cách")

## 14. Giải thích tính đúng + Big-O

**Vì sao đúng:** ta chỉ tăng `i` khi mọi chuỗi đều có đúng ký tự `first[i]`; do đó `first[:i]` luôn là tiền tố chung. Khi gặp ký tự khác hoặc hết chuỗi, không thể có tiền tố chung dài hơn `i`, nên trả `first[:i]` là tối ưu.

Gọi `n` là số chuỗi, `L` là độ dài chuỗi **đầu**. Ta xét tối đa `L` cột, mỗi cột tối đa `n-1` so sánh ⇒ **thời gian O(n·L)** (giới hạn trên). Bộ nhớ phụ của vòng lặp là **O(1)** nếu không tính chuỗi kết quả trả về; tính cả chuỗi kết quả thì **O(p)** với `p` là độ dài tiền tố. Cách `shrink` tạo chuỗi trung gian nên tốn thêm bộ nhớ.

## 15. Sai lầm người mới thường gặp

- **Nhầm prefix và substring:** kiểm tra `"low" in "flower"` đúng nhưng `"flower".startswith("low")` sai.
- **Đọc quá chỉ số:** viết `word[i]` trước khi kiểm tra độ dài.
- **Quên xét mọi chuỗi:** so chỉ hai chuỗi đầu sẽ bỏ qua chuỗi thứ ba phá vỡ prefix.
- **Trả list thay chuỗi:** đề yêu cầu `str`, không phải vị trí hay độ dài.

In [ ]:
print("low" in "flower")
print("flower".startswith("low"))
print("Chỉ 2 chuỗi:", longest_common_prefix_vertical(["apple", "apricot"]))
print("Cả 3 chuỗi:", longest_common_prefix_vertical(["apple", "apricot", "banana"]))

## 16. Khi nào nhận ra pattern?

Dấu hiệu: **nhiều chuỗi**, cần kiểm tra **cùng phần đầu**, ký tự so tại **cùng chỉ số**. Khi làm bài tương tự, hỏi: “đã có cột đầu giống nhau chưa? điều kiện nào làm kết quả dừng ngay?”

Mở rộng: nếu cần tìm **hậu tố chung** (common suffix), ta có thể so từ cuối mỗi chuỗi. Đây là một bài khác, không phải yêu cầu LeetCode #14.

## 17. Bài tập thực hành (đừng xem đáp án ngay)

Có 5 bài tăng dần. Với mỗi bài, hãy **dự đoán** đầu ra trước khi chạy cell; lời giải nằm ở cuối.

### Bài 1 — Đoán kết quả slicing — Tự làm

`s = "python"`: thử dự đoán `s[:2]`, `s[:0]`, `s[:6]`. Viết hàm trả về 3 giá trị.

**Gợi ý cấp 1 (tư duy):** Một đoạn đầu được lấy bằng cú pháp nào?

<details><summary>Gợi ý cấp 2 — Cú pháp</summary>

Dùng `s[:k]`.

</details>

<details><summary>Gợi ý cấp 3 — Gần code</summary>

`return (s[:2], ..., ...)`.

</details>

**Khi làm xong**, bỏ dấu `#` ở lệnh `assert` gợi ý trong ô code dưới để tự kiểm tra. Đáp án đầy đủ nằm ở phần cuối notebook.

In [ ]:
def exercise_14_1(s: str):
    # TODO: trả một tuple gồm s[:2], s[:0], s[:len(s)]
    pass

# assert exercise_14_1("python") == ("py", "", "python")

### Bài 2 — Tính bằng tay — Tự làm

Viết chuỗi tiền tố chung của `["star", "start", "stair"]` và thử hàm kiểm tra.

**Gợi ý cấp 1 (tư duy):** So ký tự từ trái sang phải.

<details><summary>Gợi ý cấp 2 — Cú pháp</summary>

Dừng ngay tại ký tự đầu tiên không giống.

</details>

<details><summary>Gợi ý cấp 3 — Gần code</summary>

Ký tự đầu `s`, thứ hai `t`, thứ ba là `a` ở cả 3? Kiểm lại `stair`.

</details>

**Khi làm xong**, bỏ dấu `#` ở lệnh `assert` gợi ý trong ô code dưới để tự kiểm tra. Đáp án đầy đủ nằm ở phần cuối notebook.

In [ ]:
def exercise_14_2() -> str:
    # TODO: ghi chuỗi kết quả suy luận bằng tay
    pass

# assert exercise_14_2() == "sta"

### Bài 3 — Điền vòng lặp — Tự làm

Hoàn thiện hàm so từng cột; nếu sai khác thì trả tiền tố.

**Gợi ý cấp 1 (tư duy):** Bất biến: các cột trước i đều giống.

<details><summary>Gợi ý cấp 2 — Cú pháp</summary>

`enumerate`, `for`, `len`, `or`, `[:i]`.

</details>

<details><summary>Gợi ý cấp 3 — Gần code</summary>

Đặt `if i >= len(word) or word[i] != ch` trong vòng for lồng nhau.

</details>

**Khi làm xong**, bỏ dấu `#` ở lệnh `assert` gợi ý trong ô code dưới để tự kiểm tra. Đáp án đầy đủ nằm ở phần cuối notebook.

In [ ]:
def exercise_14_3(strs: list[str]) -> str:
    if not strs:
        return ""
    # TODO: for i, ch in enumerate(strs[0]):
    # TODO: for word in strs[1:]:
    # TODO: kiểm độ dài / so ký tự; trả strs[0][:i] nếu sai
    pass

# assert exercise_14_3(["flower", "flow", "flight"]) == "fl"

### Bài 4 — Viết lại cách thu ngắn — Tự làm

Tự cài đặt `startswith` kết hợp với `prefix[:-1]`.

**Gợi ý cấp 1 (tư duy):** Lấy từ đầu list làm ứng viên.

<details><summary>Gợi ý cấp 2 — Cú pháp</summary>

`while not word.startswith(prefix)`.

</details>

<details><summary>Gợi ý cấp 3 — Gần code</summary>

Mỗi lần sai, bớt 1 ký tự cuối prefix rồi thử lại.

</details>

**Khi làm xong**, bỏ dấu `#` ở lệnh `assert` gợi ý trong ô code dưới để tự kiểm tra. Đáp án đầy đủ nằm ở phần cuối notebook.

In [ ]:
def exercise_14_4(strs: list[str]) -> str:
    # TODO: tự cài cách shrink; không gọi hàm đã giải
    pass

# assert exercise_14_4(["abc", "ab", "a"]) == "a"

### Bài 5 — Mini challenge hậu tố chung — Tự làm

Viết `common_suffix(words)` trả hậu tố chung dài nhất, ví dụ `["walking", "talking", "king"]` -> `"king"`. Đây là **biến thể**.

**Gợi ý cấp 1 (tư duy):** Đảo mọi chuỗi để biến hậu tố thành tiền tố.

<details><summary>Gợi ý cấp 2 — Cú pháp</summary>

`s[::-1]` cho chuỗi đảo.

</details>

<details><summary>Gợi ý cấp 3 — Gần code</summary>

Tính prefix của chuỗi đảo, sau đó đảo kết quả lần nữa.

</details>

**Khi làm xong**, bỏ dấu `#` ở lệnh `assert` gợi ý trong ô code dưới để tự kiểm tra. Đáp án đầy đủ nằm ở phần cuối notebook.

In [ ]:
def exercise_14_5(words: list[str]) -> str:
    # TODO: tìm hậu tố chung dài nhất; có thể dùng slicing ngược
    pass

# assert exercise_14_5(["walking", "talking", "king"]) == "king"

## 18. Đáp án tham khảo — chỉ đọc sau khi tự làm

In [ ]:
def answer_14_1(s):
    return (s[:2], s[:0], s[:len(s)])

def answer_14_2():
    return "sta"

def answer_14_3(strs):
    if not strs:
        return ""
    for i, ch in enumerate(strs[0]):
        for word in strs[1:]:
            if i >= len(word) or word[i] != ch:
                return strs[0][:i]
    return strs[0]

def answer_14_4(strs):
    if not strs:
        return ""
    prefix = strs[0]
    for word in strs[1:]:
        while not word.startswith(prefix):
            prefix = prefix[:-1]
            if not prefix:
                return ""
    return prefix

def answer_14_5(words):
    if not words:
        return ""
    reversed_words = [s[::-1] for s in words]
    reversed_prefix = answer_14_3(reversed_words)
    return reversed_prefix[::-1]

assert answer_14_1("python") == ("py", "", "python")
assert answer_14_2() == "sta"
assert answer_14_3(["flower", "flow", "flight"]) == "fl"
assert answer_14_4(["abc", "ab", "a"]) == "a"
assert answer_14_5(["walking", "talking", "king"]) == "king"
print("Đáp án tham khảo: PASS 5/5")

## Tóm tắt — mẫu tư duy cần nhớ

**Mẫu:** chọn chuỗi đầu → kiểm tra từng cột trong các chuỗi khác → gặp sai hoặc hết chuỗi thì dừng và lấy `first[:i]`. Với `n` chuỗi, dài tối đa `L`, thời gian **O(nL)**.

## Checklist trước khi submit

- [ ] Tôi phân biệt được prefix và substring.
- [ ] Tôi đã kiểm tra `i >= len(word)` trước `word[i]`.
- [ ] Tôi hiểu vì sao trả `first[:i]` chứ không phải `first[:i+1]`.
- [ ] Tôi đã chạy đủ test và biết độ phức tạp.
- [ ] Tôi chỉ dán class Solution lên LeetCode.

**Tự kiểm tra lần cuối:** có thể giải thích lời giải bằng lời trước khi xem code? Có tự tính được ít nhất một bảng trace? Nếu chưa, quay lại phần mô phỏng.

*Notebook học tập được biên soạn theo cẩm nang tiếng Việt đính kèm; tiêu đề/đề/ràng buộc bám sát LeetCode. Những biến thể được đánh dấu là mở rộng, không phải yêu cầu đề chính.*